# Plot ancestral biogeographic states

## Loading packages

In [1]:
library("stringr")
library("phytools")
library("ggtree")
library("ggplot2")
library("ggpmisc")
library("deeptime")

Le chargement a nécessité le package : ape

Le chargement a nécessité le package : maps

ggtree v3.10.1 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan, Tommy Tsan-Yuk Lam.
ggtree: an R package for visualization and annotation of phylogenetic
trees with their covariates and other associated data. Methods in
Ecology and Evolution. 2017, 8(1):28-36. doi:10.1111/2041-210X.12628

Shuangbin Xu, Lin Li, Xiao Luo, Meijun Chen, Wenli Tang, Li Zhan, Zehan
Dai, Tommy T. Lam, Yi Guan, Guangchuang Yu. Ggtree: A serialized data
object for visualization of a phylogenetic tree and annotation data.
iMeta 2022, 1(4):e56. doi:10.1002/imt2.56

LG Wang, TTY Lam, S Xu, Z Dai, L Zhou, T Feng, P Guo, CW Dunn, BR
Jones, T Bradley, H Zhu, Y Guan, Y Jiang, G Yu. treeio: an R package
for phylogenetic tree input and output with richly annotated and
associated data. Mo

## Loading data

In [2]:
trait_model<-readRDS("ancestral_state.rds")
phy<-read.tree("Input/tree_orecto.tree")

In [3]:
trait_model

,1,2,3,4,5,6,7,17,18,20,22,42,61,node
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0,0,0,0,0,0,1,0,0,0,0,0,0,1
2,0,0,0,0,0,0,1,0,0,0,0,0,0,2
3,0,0,0,0,0,1,0,0,0,0,0,0,0,3
4,0,0,0,0,0,0,0,0,0,1,0,0,0,4
5,0,0,0,0,0,0,0,0,0,1,0,0,0,5
6,0,0,0,0,1,0,0,0,0,0,0,0,0,6
7,0,0,0,0,1,0,0,0,0,0,0,0,0,7
8,0,0,0,0,0,0,0,0,0,1,0,0,0,8
9,0,0,0,0,0,0,0,0,0,1,0,0,0,9


## Data preparation

### Preparing the tree

In [4]:
phy$tip.label <- gsub("_", " ", phy$tip.label)

### Define node and tips datasets

In [5]:
node_states_biogeo <- trait_model[(length(phy$tip.label) + 1):(2*length(phy$tip.label) - 1),]
tip_states_biogeo <- trait_model[1:length(phy$tip.label),]

### Setting color list 

In [6]:
#color_list <- c("grey", "red", "blue", "yellow", "beige", "green", "cyan", "brown", "coral","chocolate", 
#               "cornflowerblue", "deeppink", "darkorchid", "darkseagreen", "deepskyblue", "darkcyan", "darkgoldenrod")

In [7]:
color_list <- c("grey", "red", "blue", "yellow", "beige", "green", "cyan", "brown", "coral","chocolate", "cornflowerblue",  "darkorchid", "darkseagreen")

In [8]:
color_list <- setNames(color_list[1:length(unique(colnames(trait_model[,-14])))],sort(unique(colnames(trait_model[,-14]))))

### Plotting the phylogeny

In [9]:
phylo_plot <- ggtree(phy) +
           geom_tiplab(offset = 2, fontface = "italic", size = 2) +
           theme_bw() +
           theme(panel.border = element_blank(),
           legend.key = element_blank(),
           axis.ticks = element_blank(),
           axis.text.y = element_blank(),
           axis.text.x = element_blank(),
           panel.grid = element_blank(),
           panel.grid.minor = element_blank(), 
           panel.grid.major = element_blank(),
           panel.background = element_blank(),
           plot.background = element_rect(fill = "transparent",colour = NA)) + 
coord_geo(neg = T, pos = as.list(rep("bottom", 2)),
dat = list("epochs", "periods"), height = list(unit(1, "lines"), unit(1, "line")), size = list(2, 3), abbrv = "auto", skip = c("Pliocene", "Pleistocene", "Holocene"))

In [10]:
phylo_plot <- revts(phylo_plot) +
           xlim(-250, 100) + ylim(0, 40)

Scale for y is already present.
Adding another scale for y, which will replace the existing scale.


## Plot ancestral states

### Prepare ancestral states as nodes

In [11]:
pies <- nodepie(node_states_biogeo, cols=1:13, color=color_list, alpha=1)
anc <- tibble::tibble(node=as.numeric(node_states_biogeo$node), pies=pies)

In [12]:
phy_plot_anc <- phylo_plot %<+% anc
phy_plot_anc <- phy_plot_anc + geom_plot(data=td_filter(!isTip), mapping=aes(x=x,y=y, label=pies), vp.width=0.04, vp.height=0.04, hjust=0.6, vjust=0.6)

### Prepare data for tippoints

In [13]:
tips_vec <- c()

for(i in 1:nrow(tip_states_biogeo)){
    temp_tip_states_biogeo <- tip_states_biogeo[,-ncol(tip_states_biogeo)]
    tips_vec <- c(tips_vec, colnames(temp_tip_states_biogeo)[which(temp_tip_states_biogeo[i,]==1)])
}

tips_vec <- as.factor(tips_vec)
levels(tips_vec) <- c(levels(tips_vec), setdiff(rownames(tip_states_biogeo[-1]), levels(tips_vec)))
df_tips <- data.frame(Species = phy$tip.label, tips_vec)

## Plot ancestral tips

In [14]:
phy_complete <- phy_plot_anc %<+% df_tips
ASE_plot <- phy_complete + geom_tippoint(aes(color = as.character(tips_vec)), show.legend = TRUE) + 
    scale_color_manual(values = color_list, drop = TRUE, limits = names(color_list)) +
    theme(legend.position = c(0.15, 0.875)) + 
guides(color = guide_legend(ncol=4, title = "Biogeographic regions"))

Warning message:
“A numeric `legend.position` argument in `theme()` was deprecated in ggplot2
3.5.0.
ℹ Please use the `legend.position.inside` argument of `theme()` instead.”


## Save figure

In [15]:
ggsave(ASE_plot, filename = "Simple_DEC.pdf",  bg = "transparent")

Saving 6.67 x 6.67 in image
